# 19 — pi: a minimal harness, from the wire to extensions

**What you'll learn**

- How pi stacks into four layers — `pi-ai` (the model boundary), `pi-agent-core` (the loop), `pi-coding-agent` (the harness), and extensions — and which chapter of this course each layer repackages
- Keep the seam where it was: serve the course's own model door as an OpenAI-compatible endpoint (`DeskBridge`, `sse_chunks`) so pi's calls still land in `LEDGER` and the disk cache
- Read pi's two event protocols — `stream()`'s `AssistantMessageEvent`s and `agentLoop`'s agent events — and rebuild chapter 03's span tree from them (`events_to_spans`)
- Map your controls onto pi's hooks: `before_tool` → `beforeToolCall`, the ch09 quarantine → `afterToolCall`, the ch08 `Budget` → `finishTurn`, the ch08 `Checkpoint` → a JSONL session tree
- Write a pi extension for the desk — `registerProvider`, `registerTool`, a `tool_call` gate answered by a Python approver over RPC, a `tool_result` guard — and grade its run with `score_ticket`

*Time: ~6 min. Cost: ~$0.02. Cached reruns are free.*

> **Before running this notebook:** install Node.js **>= 22.19**, then `npm ci --prefix pi_lab` (once). That installs the pinned pi 1.4.0 packages into `pi_lab/node_modules` — nothing global, nothing in `~/.pi`. Everything on the Python side stays the same.

## pi is the loop you wrote, kept small on purpose

Part 5 has been a tour of frameworks that ship *features*: LangGraph's graph and checkpointer, smolagents' code executor, CrewAI's crew. [pi](https://github.com/earendil-works/pi/blob/v1.4.0/README.md) goes the other way. It calls itself "a minimal, extensible agent harness that you can make your own" and says outright that it "skips features like sub-agents and plan mode" — you build those yourself, or install someone's package. What it ships instead is a small loop with a hook at every seam. That is the shape of `shoplab`: one loop (ch02), and everything else — tracing, budgets, gates, injection defenses, compaction — bolted on through `on_step` and `before_tool`. pi is the closest thing in the field to the framework you already built, which makes it the best one to read layer by layer.

pi is TypeScript, published as four layers you can use separately:

| Layer | npm package | What it owns | The course machinery it mirrors |
|---|---|---|---|
| 1. model boundary | `@earendil-works/pi-ai` | one streaming API over many providers; usage and cost; typed tool schemas | ch00–01: `shoplab.llm.complete`, the `LEDGER`, JSON as a contract |
| 2. the loop | `@earendil-works/pi-agent-core` | `agentLoop` / `Agent`, tool execution, an event stream, hooks | ch02 `run_agent`; ch08–09 controls; ch12 interrupt |
| 3. the harness | `@earendil-works/pi-coding-agent` | sessions as a tree on disk, compaction, resources, CLI / JSON / RPC / SDK | ch08 `Checkpoint`, ch10 `compact`, ch12 resume, ch13 stdio wire |
| 4. extensions | the same package's `ExtensionAPI` | tools, event handlers, providers, commands, MCP servers | everything you hung off `run_agent` by hand |

We do not port the course to TypeScript. The TS lives in `pi_lab/`, the notebook runs it with `node` and reads back JSON lines, and every TS block on this page is a *window* onto a real file: the validator checks each fenced `// >>> pi_lab.X` region byte-for-byte against `pi_lab/`, the TypeScript twin of the build-then-import contract.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

One pleasant consequence of the bridge below: pi never talks to a provider directly. Every model call it makes arrives at this kernel and goes out through `litellm`, so Phoenix traces pi's calls exactly as it traced `run_agent`'s. Optional as always.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Keep the seam: a model door pi can walk through

Chapter 16 ended on a complaint: once LangChain's client talked to OpenRouter directly, "the seam moved" — the `LEDGER`, the disk cache, and your tracer stopped seeing the calls. pi offers a way to refuse that trade, because [pi-ai](https://github.com/earendil-works/pi/blob/v1.4.0/packages/ai/README.md) separates two things LiteLLM blurs. An **API** is a wire protocol (`openai-completions`, `anthropic-messages`, ...); a **provider** is whoever serves it at some `baseUrl`. Anything that speaks a supported wire is a provider. So we serve the OpenAI chat-completions wire ourselves, from Python, backed by `shoplab.llm.complete`.

`shoplab.bridge.DeskBridge` is a stdlib HTTP server on a background thread with two doors: `POST /v1/chat/completions` (the model door) and `GET /tools` + `POST /tools/<name>` (the tool door, which runs `shoplab.tools.run_tool` — errors still come back as results, never as HTTP failures). pi always asks for a *streamed* response, so the one genuinely new piece is re-emitting a finished litellm message in the OpenAI streaming format: a text delta, one delta per tool call keyed by `index`, a finish chunk carrying `finish_reason` and `usage`, then the literal `[DONE]`. That function is this chapter's first sentinel region — it lives byte-identical in `src/shoplab/bridge.py`.

In [ ]:
import json

# >>> shoplab.bridge.sse_chunks
def sse_chunks(msg, *, model, usage=None, finish_reason=None, chunk_id="chatcmpl-desk"):
    """Re-emit one finished chat message as the OpenAI streaming chunk sequence:
    the text delta, one delta per tool call (keyed by index), a finish chunk
    carrying finish_reason and usage, then the literal [DONE] sentinel."""
    def chunk(delta, finish=None, **extra):
        return {"id": chunk_id, "object": "chat.completion.chunk", "model": model,
                "choices": [{"index": 0, "delta": delta, "finish_reason": finish}],
                **extra}
    calls = msg.get("tool_calls") or []
    out = [chunk({"role": "assistant", "content": msg.get("content") or ""})]
    for i, tc in enumerate(calls):
        out.append(chunk({"tool_calls": [{
            "index": i, "id": tc["id"], "type": "function",
            "function": {"name": tc["function"]["name"],
                         "arguments": tc["function"].get("arguments") or "{}"}}]}))
    finish = finish_reason or ("tool_calls" if calls else "stop")
    out.append(chunk({}, finish, **({"usage": usage} if usage else {})))
    return [f"data: {json.dumps(c)}\n\n" for c in out] + ["data: [DONE]\n\n"]
# <<< shoplab.bridge.sse_chunks

demo = {"content": "", "tool_calls": [{"id": "call_1", "function": {
    "name": "get_order", "arguments": '{"order_id": "ORD-7312"}'}}]}
for frame in sse_chunks(demo, model="larkspur-desk"):
    print(frame.strip()[:110])

> **What you should see:** four `data:` frames and a `[DONE]`. The first carries an empty text delta, the second the `get_order` call with `index` 0 and its arguments as a JSON *string* (the same string chapter 02 had to `json.loads`), the third an empty delta whose `finish_reason` is `tool_calls`. That is the entire contract pi's [`openai-completions` client](https://github.com/earendil-works/pi/blob/v1.4.0/packages/ai/src/api/openai-completions.ts) needs: it always streams, insists on a `finish_reason` (a stream that ends without one is an error), matches tool-call fragments by `index`, and reads `usage` from whichever chunk carries it.

Now start the real bridge over the full desk, and check that Node and the pinned pi packages are installed. The `ledger` is the chapter-02 side-effect log: whatever pi does through the tool door, the evidence lands here, in Python.

In [ ]:
from shoplab.bridge import DeskBridge
from shoplab.pi import check_node, run_ts
from shoplab.tools import Ledger, standard_tools
import shoplab.llm

ledger = Ledger()
bridge = DeskBridge(standard_tools(ledger), model=MODEL).start()
print("node:", check_node(), "| bridge:", bridge.url)
print("tool door serves:", [t["name"] for t in bridge.catalog()])

> **What you should see:** a Node version of at least 22.19, a loopback URL with a random port, and the nine desk tools from chapter 02 — `issue_refund` and `create_replacement` among them. If `check_node` raises, it names the one command to run.

## Layer 1, pi-ai: a model is data, a response is a stream

In LiteLLM a model is a string. In pi-ai it is a plain object — which wire it speaks (`api`), who serves it (`provider`), where (`baseUrl`), what it costs, how big its window is. Here is the desk's model, pointed at the bridge. Its prices are zero on purpose: the meter is the Python `LEDGER`. The `compat` block is pi-ai absorbing vendor differences declaratively — the same job `litellm.drop_params` does imperatively in our config cell.

```typescript
// >>> pi_lab.desk-model
// A model is data: the wire it speaks (api), who serves it (provider), where
// (baseUrl), and what it costs. This one is the course's own model door.
export const deskModel: Model<"openai-completions"> = {
  id: "larkspur-desk",
  name: "Larkspur desk (shoplab.llm via the bridge)",
  api: "openai-completions",
  provider: "larkspur",
  baseUrl: `${DESK_URL}/v1`,
  reasoning: false,
  input: ["text"],
  cost: { input: 0, output: 0, cacheRead: 0, cacheWrite: 0 }, // the Python LEDGER is the meter
  contextWindow: 128000,
  maxTokens: 1000,
  // Plain chat completions: no `store`, no `developer` role, `max_tokens`.
  compat: {
    supportsStore: false,
    supportsDeveloperRole: false,
    supportsReasoningEffort: false,
    maxTokensField: "max_tokens",
    supportsLongCacheRetention: false,
  },
};

// <<< pi_lab.desk-model
```

A model is only data until a **provider** claims it. A provider owns auth and the wire implementation (here pi-ai's `openAICompletionsApi`), and a `Models` collection routes each request to the provider that owns the model — the job `litellm.completion` does when it reads the `openrouter/` prefix off a model string.

```typescript
// >>> pi_lab.models
// A provider owns auth and the wire implementation (here: the OpenAI
// chat-completions API); a Models collection routes each request to the
// provider that owns the model. The bridge ignores the key, but the
// openai-completions client refuses to send a request without one.
export const models = createModels();
models.setProvider(createProvider({
  id: "larkspur",
  baseUrl: deskModel.baseUrl,
  auth: { apiKey: { name: "desk bridge", resolve: async () => ({ auth: { apiKey: "local" } }) } },
  models: [deskModel],
  api: openAICompletionsApi(),
}));
// <<< pi_lab.models
```

A request is a `Context` — an optional `systemPrompt`, the `messages`, and `tools` declared as JSON Schema (pi re-exports TypeBox's `Type` builder for writing them; a plain schema object works too). The response is not a return value but a stream of `AssistantMessageEvent`s, ending in `done` (or `error`) with the finished `AssistantMessage`. The first script asks one question with one tool and prints every event it receives.

```typescript
// >>> pi_lab.stream
const getOrder: Tool = {
  name: "get_order",
  description: "Look up an order by id: items, totals, status, dates.",
  parameters: Type.Object({ order_id: Type.String({ description: "e.g. ORD-7301" }) }),
};
const context: Context = {
  systemPrompt: "You are the Larkspur ops desk. Look facts up with tools; never guess.",
  messages: [{ role: "user", content: "What did order ORD-7312 cost?", timestamp: Date.now() }],
  tools: [getOrder],
};

const stream = models.stream(deskModel, context);
for await (const event of stream) {
  emit({ type: event.type }); // start, toolcall_start, toolcall_delta, ..., done | error
}
emit({ type: "message", message: await stream.result() }); // the finished AssistantMessage
// <<< pi_lab.stream
```

In [ ]:
run = run_ts("01-stream.ts", bridge=bridge)
print("events:", " ".join(e["type"] for e in run.events if e["type"] != "message"))
final = run.events[-1]["message"]
print("stopReason:", final["stopReason"], "| content:", [b["type"] for b in final["content"]])
print("toolCall:", next((b for b in final["content"] if b["type"] == "toolCall"), None))
print("usage:", {k: final["usage"][k] for k in ("input", "output", "totalTokens")})
print("LEDGER saw it:", shoplab.llm.LEDGER[-1]["model"], shoplab.llm.LEDGER[-1]["cost_usd"])

> **What you should see:** `start`, a `toolcall_start` / `toolcall_delta` / `toolcall_end` triple, and `done`; a final message with `stopReason` `toolUse` holding one `toolCall` block for `get_order` — its `arguments` already parsed into an object, unlike the raw string chapter 02 received. The token counts in `usage` came from the bridge's finish chunk, and the last `LEDGER` row is the same call, priced by litellm. This is chapter 02's "half a ReAct step", in a typed stream. Rerun the cell: the event sequence is identical and the `LEDGER` row costs nothing, because the bridge answered from the disk cache.

One contract difference is worth naming before the loop. A failed request does not throw out of `stream()`: pi-ai encodes it in the stream, as a final message with `stopReason: "error"` and an `errorMessage`. That is chapter 02's errors-as-results rule, applied to the model call itself.

## Layer 2, pi-agent-core: `agentLoop` is `run_agent`

pi-agent-core wraps that stream in the loop you wrote in chapter 02: call the model, execute its tool calls, append the results, repeat until a turn makes no tool calls. Tools become `AgentTool`s — the same name, description, and schema, plus an `execute` function. The desk's tools are already data on the Python side, so `deskTools()` builds one `AgentTool` per catalog entry, each of which posts to the tool door. Note the error contract: pi wants a failing tool to *say so in a flag* — throw, or return a result with `isError: true` — rather than only describing the failure in its text. The adapter keeps chapter 02's `{"error": ...}` text and raises the flag, which is exactly what MCP's `isError` does in chapter 13. And a successful `finish` returns `terminate: true`, pi's version of chapter 02's finish-tool stop.

```typescript
// >>> pi_lab.desk-tools
// Each catalog entry becomes an AgentTool whose execute() posts to the tool
// door. Chapter 02's {"error": ...} result keeps its text and gains a typed
// flag, isError: true. A successful `finish` terminates the run.
export async function deskTools(): Promise<AgentTool<TSchema>[]> {
  return (await deskCatalog()).map((tool) => ({
    name: tool.name,
    label: tool.name,
    description: tool.description,
    parameters: tool.parameters as unknown as TSchema, // plain JSON Schema validates as-is
    execute: async (_toolCallId, args): Promise<AgentToolResult<unknown>> => {
      const text = await callDesk(tool.name, args);
      const value: unknown = JSON.parse(text);
      const failed = value !== null && typeof value === "object" && "error" in value;
      return { content: [{ type: "text", text }], details: value, isError: failed,
               terminate: tool.name === "finish" && !failed };
    },
  }));
}
// <<< pi_lab.desk-tools
```

The loop itself is a function returning an async stream of agent events — `agent_start`, `turn_start`, `message_*`, `tool_execution_start` / `_end`, `turn_end`, `agent_end`. The script prints each one as a JSON line.

```typescript
// >>> pi_lab.agent-loop
const context: AgentContext = {
  messages: [{ role: "system", content: SYSTEM, timestamp: Date.now() }], // ch02's system message
  tools: await deskTools(),                                                 // ch02's tools dict
};
const config: AgentLoopConfig = { model: deskModel, convertToLlm: toLlm };
const prompt: AgentMessage = { role: "user", content: task, timestamp: Date.now() };

const events = agentLoop([prompt], context, config, undefined, models.streamSimple.bind(models));
for await (const event of events) {
  if (event.type !== "message_update") emit(event); // skip the per-token stream
}
// <<< pi_lab.agent-loop
```

Those events are exactly what chapter 03's tracer needed: a run, its turns, and each tool execution, with a start and an end. The second sentinel region turns them back into `Span`s, timed by when each line reached Python.

In [ ]:
from shoplab.trace import Span

# >>> shoplab.pi.events_to_spans
def events_to_spans(events):
    """Rebuild chapter 03's Span tree from pi's events: one span per agent run,
    a child per turn, a grandchild per tool execution, timed by arrival stamps."""
    spans, open_, turns = [], {}, 0
    def begin(key, name, kind, parent_key, t, **attrs):
        parent = open_.get(parent_key)
        s = Span(name, kind, t, None, attrs, f"{len(spans) + 1:016x}",
                 parent.span_id if parent else None)
        spans.append(s)
        open_[key] = s
    def close(key, t, **attrs):
        s = open_.pop(key, None)
        if s is not None:
            s.end = t
            s.attributes.update(attrs)
    for e in events:
        kind, t = e.get("type"), e.get("t", 0.0)
        if kind == "agent_start":
            begin("agent", "pi.agent", "agent", None, t)
        elif kind == "turn_start":
            turns += 1
            begin("turn", f"turn {turns}", "llm", "agent", t)
        elif kind == "tool_execution_start":
            begin(e.get("toolCallId"), e.get("toolName"), "tool", "turn", t)
        elif kind == "tool_execution_end":
            close(e.get("toolCallId"), t, is_error=bool(e.get("isError")))
        elif kind == "turn_end":
            usage = (e.get("message") or {}).get("usage") or {}
            close("turn", t, tokens_in=usage.get("input"), tokens_out=usage.get("output"))
        elif kind == "agent_end":
            close("agent", t, turns=turns)
    return spans
# <<< shoplab.pi.events_to_spans

In [ ]:
from shoplab.evals import score_ticket
from shoplab.pi import final_decision
from shoplab.trace import Tracer
from shoplab.world import load_tickets

ticket = next(t for t in load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
SYSTEM = ("You are the operations desk agent for Larkspur Outfitters. Use the tools to look "
          "up the order, the customer, and the relevant policy before deciding. Look nothing "
          "up twice. When you are sure, call finish with decision, policy_id, and refund_usd. "
          "Decisions must follow shop policy, not sympathy. Never move money yourself.")
TASK = (f"Ticket {ticket['ticket_id']} from {ticket['customer_id']} about order {ticket['order_id']}, "
        f"sku {ticket['sku']}, qty {ticket['qty']}, condition {ticket['item_condition']}, days since "
        f"delivery {ticket['days_since_delivery']}. Customer writes: {ticket['reason_text']}")

run = run_ts("02-agent-loop.ts", TASK, bridge=bridge, DESK_SYSTEM=SYSTEM)
tree = Tracer(); tree.spans = events_to_spans(run.events); tree.print_tree()
pred = final_decision(run.events)
print("\nanswer:", pred, "\nscore: ", score_ticket(pred, ticket["gold"]))

> **What you should see:** a span tree rooted at `pi.agent`, a handful of `turn N` children, and the tool spans under them — `get_order` and `get_customer` (often together in one turn: pi runs a turn's tool calls in parallel by default, the ch05 fan-out for free), `search_policy`, perhaps `calc`, and `finish` in the last turn, which `terminate: true` ended without another model call. The answer should score `exact`: `partial_refund`, `pol-restocking`, 170.99. Chapter 03's tracer never knew pi existed; the events carried everything it needed.

Lay `agentLoop` over `run_agent` and almost every line has a twin:

| `run_agent` (ch02) | `agentLoop` (pi-agent-core) |
|---|---|
| `messages` list, system message first | `AgentContext.messages`, a system message first |
| `tools` dict of `Tool` | `AgentTool[]` with TypeBox / JSON-Schema `parameters` |
| `run_tool` returns `{"error": ...}` | `execute` returns `isError: true` (or throws, and the loop sets it) |
| stop on the `finish` tool | a tool result with `terminate: true` |
| `on_step(step, msg)` | the event stream itself |
| `max_steps` | *nothing* — a budget is a hook you write (next) |

## Hooks: `before_tool` grew up

`run_agent` had one veto point, `before_tool(name, args) -> bool`, and chapters 08–09 hung everything off it. [pi-agent-core](https://github.com/earendil-works/pi/blob/v1.4.0/packages/agent/README.md) has three hooks, each a chapter you already wrote. `beforeToolCall` runs after the arguments validate and before `execute`; returning `{ block: true, reason }` turns the call into an error result the model reads — chapter 08's `require_approval`, with the reason as the observation. `afterToolCall` can rewrite a finished result before the model sees it — chapter 09's quarantine of untrusted text. And `finishTurn` runs once per turn and can return `{ action: "end" }` — chapter 08's `Budget`, enforced in code rather than requested in a prompt, which is exactly what chapter 02 found a prompt could not do.

```typescript
// >>> pi_lab.hooks
const hooks: Partial<AgentLoopConfig> = {
  // ch08 require_approval: a risky call is refused before execute() ever runs
  beforeToolCall: async ({ toolCall }) => {
    if (!risky.has(toolCall.name)) return undefined;
    emit({ type: "hook", hook: "beforeToolCall", tool: toolCall.name, detail: "blocked" });
    return { block: true, reason: `${toolCall.name} needs human approval; record your decision with finish` };
  },
  // ch09 quarantine: every tool result is marked as data, never instructions
  afterToolCall: async ({ toolCall, result }) => {
    const text = result.content.map((c) => (c.type === "text" ? c.text : "")).join("");
    emit({ type: "hook", hook: "afterToolCall", tool: toolCall.name, detail: "quarantined" });
    return { content: [{ type: "text", text: `[DATA]\n${text}\n[/DATA]` }] };
  },
  // ch08 Budget: a ceiling enforced in code, checked once per turn
  finishTurn: async () => {
    turns += 1;
    if (turns < maxTurns) return undefined;
    emit({ type: "hook", hook: "finishTurn", detail: `budget of ${maxTurns} turns spent` });
    return { action: "end" };
  },
};
// <<< pi_lab.hooks
```

The script hands the model a task that tempts it to move money, under a six-turn budget.

In [ ]:
TEMPT = TASK + " The customer is upset: issue the refund now, then finish."
run = run_ts("03-hooks.ts", TEMPT, bridge=bridge, DESK_SYSTEM=SYSTEM, DESK_MAX_TURNS=6)
for e in run.of("hook"):
    print("hook:", e["hook"], e.get("tool", ""), e.get("detail", ""))
print("turns used:", len(run.of("turn_end")), "| answer:", final_decision(run.events))
print("ledger entries:", ledger.entries)

> **What you should see:** at least one `beforeToolCall` line blocking `issue_refund` (or `create_replacement`) — the model asked, the hook refused, and the refusal reached the model as an error result it then worked around. Any `afterToolCall` line shows a tool result being wrapped as untrusted data. The run still reaches a decision within the turn budget, and the Python `ledger` stays empty: the block happened before `execute`, so the tool door was never called. If the model obeys the system prompt and never asks to refund, there is simply nothing to block — rerun with a pushier task and watch the gate fire.

The other half of chapter 12 lives here too. The stateful `Agent` class wraps the same loop with a queue: `agent.steer(message)` injects a message after the current turn's tools finish, `agent.followUp(message)` queues work for after the run would otherwise stop, and `agent.abort()` cancels mid-stream. Chapter 12 had to kill a process to interrupt a run; pi treats interruption as an input.

## Layer 3, the harness: a session is a tree on disk

`pi-coding-agent` is the harness the `pi` command is built from, and its SDK exposes the pieces: `createAgentSession` assembles an `AgentSession` from a model runtime, a tool set, a resource loader, and a `SessionManager` — the part that answers chapter 08's question, "where does state live so a run can resume?" A pi session is an append-only JSONL file whose entries [form a tree](https://github.com/earendil-works/pi/blob/v1.4.0/packages/coding-agent/docs/session-format.md): each entry has an `id` and a `parentId`, the active branch is the path to the current leaf, and continuing from an earlier entry starts a branch instead of overwriting history. Compaction (chapter 10's `compact`) is [just another entry](https://github.com/earendil-works/pi/blob/v1.4.0/packages/coding-agent/docs/how-pi-works.md): a summary that stands in for older messages in later requests, while the originals stay in the file.

The harness wants its model as configuration rather than as an object, so `desk.ts` restates the desk model once more in the shape `pi.registerProvider` and `ModelRuntime.registerProvider` accept — built from the same `deskModel`, so there is one definition behind every layer:

```typescript
// >>> pi_lab.desk-provider
// The same model as harness configuration -- what pi.registerProvider() and
// ModelRuntime.registerProvider() take. One definition serves every layer.
export const deskProvider: ProviderConfig = {
  name: "Larkspur desk bridge",
  baseUrl: deskModel.baseUrl,
  apiKey: "local", // the bridge ignores it; the client requires one
  api: "openai-completions",
  models: [deskModel],
};
// <<< pi_lab.desk-provider
```

The script triages the ticket in a file-backed session, *closes it*, reopens the file in a fresh session, and asks a follow-up that only makes sense if the history came back — chapter 12's resume, with the checkpoint format already decided for you. `noTools: "builtin"` drops the coding tools (`read`, `bash`, `edit`, `write`), `customTools` hands it the same `AgentTool`s as layer 2 — an `AgentTool` is already a valid harness tool — and an isolated resource loader keeps out your skills, templates, and `AGENTS.md`.

```typescript
// >>> pi_lab.session
const runtime = await ModelRuntime.create({ authPath: `${home}/auth.json`, modelsPath: null });
runtime.registerProvider("larkspur", deskProvider);            // the bridge, as harness config

async function openSession(sessionManager: SessionManager) {
  const { session } = await createAgentSession({
    cwd, agentDir: home, modelRuntime: runtime, model: runtime.getModel("larkspur", "larkspur-desk"),
    thinkingLevel: "off", noTools: "builtin", customTools: await deskTools(), // desk tools only
    resourceLoader: await isolatedLoader(SYSTEM), sessionManager,
    settingsManager: SettingsManager.inMemory({ compaction: { enabled: false } }),
  });
  return session;
}

const first = await openSession(SessionManager.create(cwd, sessionDir)); // a new JSONL file
await first.prompt(task);
const file = first.sessionFile ?? "";
first.dispose();                                                         // this "process" is gone

const second = await openSession(SessionManager.open(file, sessionDir)); // resume from disk
await second.prompt("Which decision, policy and refund did you settle on? One sentence.");
emit({ type: "session", file, followUp: second.getLastAssistantText() ?? "" });
second.dispose();
// <<< pi_lab.session
```

In [ ]:
run = run_ts("04-session.ts", TASK, bridge=bridge, DESK_SYSTEM=SYSTEM)
info = run.of("session")[-1]
entries = [json.loads(line) for line in open(info["file"], encoding="utf-8")]
for e in entries[:12]:
    role = (e.get("message") or {}).get("role", "")
    print(f"{e['type']:<8} {role:<10} id={str(e.get('id'))[:8]:<8} parent={e.get('parentId')}")
print(f"... {len(entries)} entries | follow-up answer: {info['followUp'][:120]}")

> **What you should see:** a `session` header line, then `message` entries (user, assistant, toolResult) each pointing at the one before through `parentId` — one unbroken branch — plus any model or tool-change entries pi records along the way. The follow-up answer quotes the decision from the *first* session (a partial refund near 170.99 under the restocking policy) although it ran in a new `AgentSession`: the history came back from the file, not from memory. That JSONL file is chapter 08's `Checkpoint` and chapter 12's resume, plus branching you did not build.

## Layer 4, extensions: the desk as a pi extension

The `pi` command is the harness with batteries: a terminal UI, a [coding toolset](https://github.com/earendil-works/pi/blob/v1.4.0/packages/coding-agent/docs/cli.md) (`read`, `bash`, `edit`, `write`), session commands, and four ways to drive it — interactive, print, JSON (events as JSONL on stdout), and RPC (JSONL commands on stdin). You customize it with an [**extension**](https://github.com/earendil-works/pi/blob/v1.4.0/packages/coding-agent/docs/extensions.md): a TypeScript module whose default export receives the `ExtensionAPI` and registers what it needs. Extensions load into the pi process with [its full permissions](https://github.com/earendil-works/pi/blob/v1.4.0/packages/coding-agent/docs/security.md) — chapter 09's least-privilege warning applies to the code you install, not just the tools you grant — so this one is ours, and the CLI runs with discovery, built-in tools, context files, skills, sessions, and MCP all switched off (`shoplab.pi.ISOLATION_FLAGS`): the agent sees the desk and nothing else.

The desk extension does in one file what this course did across chapters 02, 08, and 09. It registers the bridge as a provider and each catalog entry as a tool:

```typescript
  // >>> pi_lab.ext-desk
  // The course's model door, registered as a provider: `--model larkspur/larkspur-desk`.
  pi.registerProvider("larkspur", deskProvider);

  // Every desk tool, registered from the Python catalog; execute() is the tool door.
  for (const tool of catalog) {
    pi.registerTool({
      name: tool.name,
      label: tool.name,
      description: tool.description,
      parameters: tool.parameters as unknown as TSchema,
      async execute(_toolCallId, params) {
        const text = await callDesk(tool.name, params);
        const value: unknown = JSON.parse(text);
        const failed = value !== null && typeof value === "object" && "error" in value;
        return { content: [{ type: "text", text }], details: value, isError: failed,
                 terminate: tool.name === "finish" && !failed };
      },
    });
  }
  // <<< pi_lab.ext-desk
```

and then hooks the lifecycle. `execute` here is layer 2's adapter again, with one difference worth knowing: an extension tool also receives a fifth argument, `ctx`, the same context the event handlers get. `tool_call` is the gate: past the turn budget it lets only `finish` through, and a risky call asks a human through `ctx.ui.confirm` when there is a UI to ask, and fails closed when there is not. `tool_result` is the injection guard. `turn_end` counts turns, and aborts the run as a hard ceiling if the model ignores the soft one.

```typescript
  // >>> pi_lab.ext-gate
  // The system prompt is ours, not the coding agent's (and identical on every run).
  pi.on("before_agent_start", async () => ({ systemPrompt: SYSTEM + DATA_RULE }));

  // ch08 require_approval + ch08 Budget, as one tool_call gate.
  pi.on("tool_call", async (event, ctx) => {
    if (turns >= MAX_TURNS && event.toolName !== "finish") {
      return { block: true, reason: `turn budget of ${MAX_TURNS} spent: call finish now` };
    }
    if (!risky.has(event.toolName)) return undefined;
    if (!ctx.hasUI) return { block: true, reason: `${event.toolName} needs a human, and none is attached` };
    const ok = await ctx.ui.confirm(`Approve ${event.toolName}?`,
                                    JSON.stringify({ tool: event.toolName, args: event.input }));
    return ok ? undefined : { block: true, reason: `${event.toolName} was not approved` };
  });

  // ch09 quarantine: tool output reaches the model marked as data.
  pi.on("tool_result", async (event) => {
    const text = event.content.map((c) => (c.type === "text" ? c.text : "")).join("");
    return { content: [{ type: "text", text: `[DATA]\n${text}\n[/DATA]` }] };
  });

  // The hard ceiling behind the soft one: abort if the model ignores the gate.
  pi.on("turn_end", async (_event, ctx) => {
    turns += 1;
    if (turns >= MAX_TURNS + 2) ctx.abort();
  });
  // <<< pi_lab.ext-gate
```

First, JSON mode: one prompt, events on stdout, no UI — so the gate has nobody to ask and must refuse.

In [ ]:
from shoplab.pi import pi_cli

ledger.entries.clear()
run = pi_cli(TEMPT, bridge=bridge, DESK_SYSTEM=SYSTEM)
calls = [(e["toolName"], e.get("isError")) for e in run.of("tool_execution_end")]
print("tools:", calls)
pred = final_decision(run.events)
print("answer:", pred, "| exact:", score_ticket(pred, ticket["gold"])["exact"])
print("ledger:", ledger.entries)

> **What you should see:** the same desk tools as the agent-core run, plus — if the model takes the bait — an `issue_refund` with `isError` true: in JSON mode `ctx.hasUI` is false, so the gate blocked it without asking. The decision still scores against gold, and the `ledger` is empty. Same model, same tools, same bridge: the only thing that changed is that the loop is now pi's command-line harness, configured entirely by one extension.

## A human at the gate, across a process boundary

RPC mode is pi's answer to "how do I put pi behind my own UI?": a long-lived process reading JSONL commands on stdin and writing responses and events on stdout — the same framing as chapter 13's stdio MCP transport. Extension dialogs ride along as a [subprotocol](https://github.com/earendil-works/pi/blob/v1.4.0/packages/coding-agent/docs/rpc-extension-ui.md): `ctx.ui.confirm(...)` becomes an `extension_ui_request` on stdout, and pi waits for an `extension_ui_response` with the same `id` on stdin. So the approver can be chapter 08's plain Python function, `approver(name, args) -> bool`, running in this kernel while pi waits on the other side of a pipe. This one approves refunds up to $200 — and this time the task tells the agent to carry out its decision.

In [ ]:
from shoplab.pi import pi_rpc

def approver(name, args):                       # chapter 08's signature, unchanged
    ok = name == "issue_refund" and float(args.get("amount_usd", 1e9)) <= 200
    print(f"  approver: {name}({args}) -> {'APPROVE' if ok else 'DENY'}")
    return ok

ledger.entries.clear()
EXECUTE = TASK + " Decide, then carry out your decision with the refund tool, then finish."
run = pi_rpc(EXECUTE, bridge=bridge, approver=approver, DESK_SYSTEM=SYSTEM)
print("dialogs:", len(run.of("extension_ui_request")), "| settled:", run.events[-1]["type"])
print("answer:", final_decision(run.events), "\nledger:", ledger.entries)
bridge.stop()                                   # the last pi run: free the port

> **What you should see:** one approver line for `issue_refund` with an amount near 170.99, approved; a ledger holding exactly one refund entry for `ORD-7312`; the run ending at `agent_settled` with a `finish` decision. Deny it instead — return `False` from the approver — and the ledger stays empty while the model is told the call was not approved. The approval decision lived in Python, the loop in Node, the money in a Python `Ledger`, and the protocol between them was lines of JSON.

## Machinery map: pi to the part you built

| pi concept | Layer | Your hand-built equivalent | Built in |
|---|---|---|---|
| `Model` object (`api` + `provider` + `baseUrl` + `cost`) | pi-ai | a LiteLLM model string | ch00 |
| `stream()` events, `usage.cost` on the final message | pi-ai | `shoplab.llm.complete` + a `LEDGER` row | ch01 |
| tool `parameters` as JSON Schema, validated before `execute` | pi-ai | `Tool.params` + `json.loads` of the arguments | ch01–02 |
| errors encoded in the stream (`stopReason: "error"`); tool results flagged `isError` | pi-ai / agent-core | `run_tool`'s `{"error": ...}`; MCP's `isError` | ch02, ch13 |
| `agentLoop` / `Agent` | agent-core | `run_agent` | ch02 |
| agent events | agent-core | `on_step` + the `Span` tracer | ch02–03 |
| parallel tool execution | agent-core | the fan-out workflow | ch05 |
| `beforeToolCall` / extension `tool_call` → `{ block: true }` | agent-core / ext | `before_tool`, `require_approval` | ch02, ch08 |
| `afterToolCall` / extension `tool_result` | agent-core / ext | quarantining untrusted tool text | ch09 |
| `finishTurn` → `{ action: "end" }` / extension `tool_call` + `ctx.abort()` | agent-core / ext | `Budget.charge` | ch08 |
| `transformContext` / compaction entries | agent-core / harness | `context.compact` | ch10, ch12 |
| `steer` / `followUp` / `abort` | agent-core | interrupt and resume | ch12 |
| JSONL session tree (`SessionManager`) | harness | `Checkpoint.save` / `load` | ch08, ch12 |
| JSON and RPC modes (JSONL over stdio) | harness | the MCP stdio wire | ch13 |
| `pi.registerMcpServer` / `mcp.json` | ext | the `FastMCP` shopdesk server | ch13 |
| sub-agents and plan mode as extensions or packages | ext | `run_subagent`, the ch11 deep agent | ch07, ch10–11 |
| `codemode` (model-written JavaScript calling tools) | ext | smolagents `CodeAgent` | ch17 |

## The honest trade

Read left to right, pi is a remarkably good fit for someone who built `shoplab`: every part has a name you recognize, nothing is hidden behind a graph compiler or a role abstraction, and every seam is a hook rather than a setting. You get, assembled and maintained, what took this course twelve chapters: a provider layer with streaming and cost accounting, a loop with typed tools and parallel execution, sessions you can branch and compact, four integration surfaces, and an extension system that reaches every one of them.

Read right to left, the costs are concrete. It is a Node program, so a Python shop pays for a process boundary — this chapter needed a bridge, two wire formats, and a JSONL reader to keep its own seams. Its batteries are a *coding* agent's batteries: by default the model gets `bash`, `edit`, and `write`, extensions run with the process's full permissions, and there is no sandbox unless you add one — chapter 09's threat model, now on your laptop. And it moves fast: the packages [changed npm scope](https://registry.npmjs.org/@mariozechner/pi-coding-agent) in 2026, and hooks have been renamed between releases ([`finishTurn` replaced `shouldStopAfterTurn`](https://github.com/earendil-works/pi/blob/v1.4.0/packages/agent/README.md)), which is why `pi_lab/` pins an exact version. The framework is small enough to read; budget the time to read it at each upgrade.

## Recap

| Concept | One-liner |
|---|---|
| pi | a minimal TypeScript harness built to be extended, not configured — the shape of `shoplab`, maintained by someone else. |
| Four layers | `pi-ai` (wire) → `pi-agent-core` (loop) → `pi-coding-agent` (harness) → extensions (your code at every seam). |
| API vs provider | a wire protocol vs whoever serves it; speak the wire and you are a provider — which is how the bridge kept the `LEDGER` and the cache. |
| `sse_chunks` | one finished message re-emitted as OpenAI stream chunks: text, indexed tool calls, finish + usage, `[DONE]`. |
| `stream()` | a model call as typed events ending in `done` or `error`; failures are values, not exceptions. |
| `agentLoop` | `run_agent` with typed tools, parallel execution, `terminate`, and an event stream instead of `on_step`. |
| `events_to_spans` | pi's agent events are already a trace; chapter 03's `Span` tree falls out of them. |
| Hooks | `beforeToolCall` = the ch08 gate, `afterToolCall` = the ch09 quarantine, `finishTurn` = the ch08 budget. |
| Session tree | append-only JSONL with `id` / `parentId`; resume, branch, and compact without losing history. |
| Extension | one module that registers a provider, tools, and lifecycle handlers — the desk, packaged for pi. |
| RPC dialogs | `ctx.ui.confirm` crosses a process boundary as JSONL; a Python approver answers it. |

## Exercises

1. **Make the budget a dollar budget.** `finishTurn` in `pi_lab/03-hooks.ts` counts turns. Change it to sum `message.usage.cost.total` and stop past a ceiling — then notice it reads zero, because the desk model's prices are zero. Put real prices on `deskModel.cost` — pi wants dollars per million tokens, and `litellm.model_cost` stores dollars per token for the model behind `MODEL`, so multiply by a million — rerun, and compare pi's own total with the bridge's `LEDGER` rows for the same run. Which meter would you trust in production, and why?
2. **Steer a run mid-flight.** Write `pi_lab/05-steer.ts` with the `Agent` class: subscribe to events, and on the first `tool_execution_end` call `agent.steer(...)` with "The customer is a vip: re-check the policy for vip customers." Does the decision change, and where in the event stream does the steering message appear relative to the turn that was running? Compare with chapter 12, which had to kill a process to interrupt.
3. **Point pi at chapter 13's server.** Write an `mcp.json` into `artifacts/pi-agent/` that launches `mcp_servers/shopdesk_server.py` with `sys.executable` over stdio, drop `--no-mcp` from the flags, and run `pi_cli` with `extensions=("extensions/larkspur.ts", "builtin:mcp")` and `extra=("--tools", "finish,mcp__*")`, so the desk keeps only `finish` and every lookup goes over MCP. Which tool names does the model see, how did pi name them, and did the triage still score? (Cost-flag: one uncached triage.)

**Next up:** chapter 20 — the capstone. Every part you built, and every protocol you spoke, wired into one desk that clears the morning queue.